> **제12회 기출변형 문제** — 교재 560~590쪽
> 문제 560~562쪽 / 정답·해설 563~590쪽
>
> 노트북과 같은 위치에 데이터 파일(csv)을 두고 위에서부터 차례대로 실행하세요.

## 문제

제공된 데이터(equip_train.csv)는 산업설비의 온도, 가동시간, IoT 센서 측정값에 따른 설비 유지비용 데이터이다. 학습 데이터(equip_train.csv)를 이용하여 RandomForest, LightGBM, XGBoost 모형을 구축하고, 평가 데이터(equip_test.csv)에 대한 예측 결과에 대해 RMSLE(Root Mean Squared Logarithmic Error, 평균제곱로그오차 제곱근)를 서로 비교하시오. 그리고 RandomForest 모형에 대해 하이퍼파라미터 자동 최적화 기법을 이용하여 최적의 모델을 구축하시오. 단, 모든 결과는 반올림하여 소수점 넷째 자리까지 출력한다.

```
RMSLE = sqrt( (1/n) * Σ(log(y_i + 1) - log(ŷ_i + 1))² )
```
- n: 전체 데이터 개수
- y_i: i번째 데이터의 실제값(Actual Value)
- ŷ_i: i번째 데이터의 예측값(Predicted Value)

**제공 데이터**
- equip_train.csv: 학습용 데이터, 800개
- equip_test.csv: 평가용 데이터, 300개 (둘 다 cost 컬럼 포함)

| 컬럼명 | 설명 |
|---|---|
| id | 설비 번호 |
| temp | 설비 온도 |
| runtime | 사용시간 |
| sensor1 ~ sensor20 | IoT 센서 측정값 |
| cost | 유지비용 |

### 문제 풀이 방향

지금까지의 2유형 문제들은 학습용 데이터에만 정답이 있고 평가용 데이터의 정답을 예측해서 제출하는 형태였지만, 이 문제는 조금 다릅니다. equip_test.csv에도 실제 cost 값이 포함되어 있고, 이 값을 이용해 여러 모델의 성능을 직접 비교하는 것이 문제의 핵심입니다. 즉 제출 파일을 만드는 것이 아니라, 세 가지 모델(RandomForest, LightGBM, XGBoost)을 각각 학습시키고 예측한 뒤, RMSLE라는 지표로 어느 모델이 가장 좋은지 비교하고, 가장 널리 쓰이는 RandomForest에 대해서는 하이퍼파라미터를 자동으로 탐색해서 성능을 더 끌어올리는 것까지가 이번 문제의 범위입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 구조 확인**

In [ ]:
import pandas as pd

train = pd.read_csv('equip_train.csv')
test = pd.read_csv('equip_test.csv')

print(train.shape, test.shape)
train[['id', 'temp', 'runtime', 'sensor1', 'sensor2', 'sensor3', 'cost']].head()

(800, 24) (300, 24)


,id,temp,runtime,sensor1,sensor2,sensor3,cost
0,1,85.0,7940.4,56.99,53.71,46.19,7049.1
1,2,75.7,4573.5,53.19,53.24,57.02,4415.5
2,3,41.8,4182.6,61.38,32.87,29.61,6255.0
3,4,64.9,7063.3,65.77,58.74,68.54,5870.6
4,5,74.3,194.1,61.52,49.06,49.41,5157.1


train과 test 모두 24개 컬럼(id, temp, runtime, sensor1~sensor20, cost)을 가지고 있습니다. 컬럼이 워낙 많아서 sensor4 이후는 head() 출력에서 생략하고 대표적인 컬럼만 확인했습니다. sensor 컬럼이 20개나 되지만, 이 중 실제로 비용과 관련이 깊은 센서가 몇 개인지, 나머지는 그냥 잡음에 가까운 변수인지는 모델을 학습시켜본 뒤에야 짐작할 수 있습니다.

**셀 2) 독립변수(X)와 종속변수(y) 구성**

In [ ]:
X_train = train.drop(columns=['id', 'cost'])
y_train = train['cost']
X_test = test.drop(columns=['id', 'cost'])
y_test = test['cost']

print(X_train.shape, X_test.shape)

(800, 22) (300, 22)


id는 설비를 구분하는 값일 뿐이고, cost는 우리가 맞히고 싶은 종속변수이므로 둘 다 X에서 제외했습니다. temp, runtime, sensor1~sensor20을 합쳐 총 22개의 독립변수가 남습니다.

**셀 3) RandomForest 모형 학습 및 RMSLE 계산**

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_log_error
from sklearn.ensemble import RandomForestRegressor

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_rmsle = round(rmsle(y_test, rf_pred), 4)

print("RandomForest RMSLE:", rf_rmsle)

RandomForest RMSLE: 0.1871


RMSLE는 사이킷런에 기본 함수로 준비되어 있지 않으므로, `mean_squared_log_error`(로그오차의 제곱평균, MSLE)를 구한 뒤 직접 제곱근을 씌워서 계산하는 `rmsle()` 함수를 만들었습니다. RMSLE 공식에는 `log(예측값+1)`이 들어가는데, 로그 함수는 0 이하의 값에 대해 정의되지 않으므로, 혹시 모델이 음수를 예측하는 경우에 대비해 `np.clip(y_pred, 0, None)`으로 예측값의 하한을 0으로 미리 보정해두었습니다.

> 💡 RMSLE는 RMSE와 달리 로그를 취한 뒤 오차를 계산하기 때문에, 값이 큰 데이터의 절대적인 오차보다 상대적인(비율) 오차에 더 집중하는 지표입니다. 예를 들어 실제값이 100인데 90으로 예측한 경우와, 실제값이 10,000인데 9,000으로 예측한 경우, RMSE 관점에서는 오차 크기(10 vs 1,000)가 완전히 다르지만 RMSLE 관점에서는 두 경우 모두 "10% 정도 낮게 예측했다"는 점에서 비슷하게 취급됩니다. 유지비용처럼 설비마다 금액 규모 차이가 클 수 있는 데이터에서 RMSLE가 종종 사용되는 이유입니다.

**셀 4) LightGBM 모형 학습 및 RMSLE 계산**

In [ ]:
# lightgbm 설치
%pip install lightgbm -q

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
lgbm_pred = lgbm.predict(X_test)
lgbm_rmsle = round(rmsle(y_test, lgbm_pred), 4)

print("LightGBM RMSLE:", lgbm_rmsle)

LightGBM RMSLE: 0.1967


LightGBM은 `lgb.LGBMRegressor()`로 모델을 만듭니다. `verbose=-1`은 학습 과정에서 나오는 상세 로그 메시지를 출력하지 않도록 끄는 옵션으로, 결과 확인에 불필요한 출력을 줄여줍니다. RandomForest와 사용법(`.fit()`, `.predict()`)이 거의 동일하다는 것을 확인할 수 있습니다.

**셀 5) XGBoost 모형 학습 및 RMSLE 계산**

In [ ]:
# XGBoost 설치
!pip install xgboost -q

In [ ]:
import xgboost as xgb

xgbm = xgb.XGBRegressor(random_state=42, verbosity=0)
xgbm.fit(X_train, y_train)
xgb_pred = xgbm.predict(X_test)
xgb_rmsle = round(rmsle(y_test, xgb_pred), 4)

print("XGBoost RMSLE:", xgb_rmsle)

XGBoost RMSLE: 0.2068


XGBoost도 `xgb.XGBRegressor()`로 만들고 같은 방식(`.fit()`, `.predict()`)으로 학습·예측합니다. `verbosity=0`은 LightGBM의 `verbose=-1`과 같은 역할로, 불필요한 로그 출력을 막습니다. 세 모델 모두 트리 기반 앙상블이라는 공통점이 있어서, sklearn 스타일의 인터페이스(`.fit()`, `.predict()`)가 거의 동일하게 유지된다는 것이 편리한 점입니다.

**셀 6) 세 모델의 RMSLE 비교**

In [ ]:
compare = pd.DataFrame({
    'model': ['RandomForest', 'LightGBM', 'XGBoost'],
    'RMSLE': [rf_rmsle, lgbm_rmsle, xgb_rmsle]
}).sort_values('RMSLE')

print(compare)

          model   RMSLE
0  RandomForest  0.1871
1      LightGBM  0.1967
2       XGBoost  0.2068


세 모델의 RMSLE를 한 표에 모아 오름차순으로 정렬했습니다. RMSLE는 오차 지표이므로 값이 작을수록 좋은 모델입니다. 이 데이터에서는 RandomForest가 가장 낮은 RMSLE(0.1871)를 기록해 세 모델 중 가장 우수한 성능을 보였습니다.

**셀 7) RandomForest 하이퍼파라미터 자동 최적화**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

print("최적 하이퍼파라미터:", random_search.best_params_)

최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': None}


하이퍼파라미터란 모델이 데이터로부터 스스로 학습하는 값이 아니라, 학습을 시작하기 전에 사람이 미리 정해줘야 하는 설정값입니다(트리 개수, 트리 깊이 등). `RandomizedSearchCV`는 미리 정해둔 후보값 목록(`param_dist`) 중에서 무작위로 몇 가지 조합(`n_iter=20`이므로 20가지 조합)을 뽑아 성능을 비교하고, 그 중 가장 좋은 조합을 자동으로 찾아주는 도구입니다.

`cv=3`은 각 조합의 성능을 평가할 때 데이터를 3등분해서 교차검증하겠다는 뜻입니다. 한 번만 검증하면 우연히 운이 좋거나 나쁜 결과가 나올 수 있는데, 데이터를 여러 조각으로 나눠 돌아가며 검증하면 더 안정적인 평가가 가능합니다. `scoring='neg_mean_squared_error'`는 평가 기준으로 MSE(작을수록 좋음)의 부호를 반대로 뒤집은 값을 사용한다는 뜻인데, 이는 사이킷런의 탐색 도구들이 관례적으로 "점수는 클수록 좋다"는 규칙을 따르기 때문에, 원래는 작을수록 좋은 오차 지표를 음수로 바꿔서 "음수 오차가 클수록(즉 오차가 작을수록) 좋은 점수"로 취급하기 위한 것입니다. `n_jobs=-1`은 컴퓨터가 가진 모든 코어를 동시에 사용해서 여러 조합을 병렬로 계산하라는 뜻으로, 탐색 속도를 높여줍니다.

`RandomizedSearchCV`가 `GridSearchCV`와 다른 점은, GridSearchCV가 모든 조합을 하나도 빠짐없이 전부 시도하는 반면 RandomizedSearchCV는 무작위로 일부(`n_iter`개)만 시도한다는 것입니다. 이번 후보 목록은 4×5×3×3=180가지 조합이 가능한데, 이걸 전부 다 시도하는 대신 20가지만 무작위로 뽑아 시도했으므로 훨씬 빠르게 끝납니다.

> 💡 후보 조합의 수가 많을 때는 GridSearchCV(전수 탐색)보다 RandomizedSearchCV(무작위 탐색)가 훨씬 빠르면서도 실무에서는 비슷한 수준의 좋은 조합을 찾아내는 경우가 많습니다. 시험처럼 시간 제약이 있는 상황에서는 RandomizedSearchCV를 우선 고려하는 것이 안전합니다.

**셀 8) 최적 모델로 재예측 및 RMSLE 확인**

In [ ]:
best_rf = random_search.best_estimator_
best_pred = best_rf.predict(X_test)
best_rmsle = round(rmsle(y_test, best_pred), 4)

print("튜닝 후 RandomForest RMSLE:", best_rmsle)

튜닝 후 RandomForest RMSLE: 0.1859


`random_search.best_estimator_`는 탐색 과정에서 찾은 최적의 하이퍼파라미터로 이미 학습까지 완료된 모델 객체입니다. 이 모델로 test 데이터를 다시 예측해 RMSLE를 계산한 결과, 튜닝 전(0.1871)보다 튜닝 후(0.1859)에 값이 더 작아져 성능이 소폭 개선된 것을 확인할 수 있습니다.

### 전체 코드

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_log_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
import lightgbm as lgb
import xgboost as xgb

train = pd.read_csv('equip_train.csv')
test = pd.read_csv('equip_test.csv')

X_train = train.drop(columns=['id', 'cost'])
y_train = train['cost']
X_test = test.drop(columns=['id', 'cost'])
y_test = test['cost']

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

# 세 모델 학습 및 비교
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
rf_rmsle = round(rmsle(y_test, rf.predict(X_test)), 4)

lgbm = lgb.LGBMRegressor(random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
lgbm_rmsle = round(rmsle(y_test, lgbm.predict(X_test)), 4)

xgbm = xgb.XGBRegressor(random_state=42, verbosity=0)
xgbm.fit(X_train, y_train)
xgb_rmsle = round(rmsle(y_test, xgbm.predict(X_test)), 4)

print("RandomForest RMSLE:", rf_rmsle)
print("LightGBM RMSLE:", lgbm_rmsle)
print("XGBoost RMSLE:", xgb_rmsle)

# RandomForest 하이퍼파라미터 튜닝
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=20, cv=3, scoring='neg_mean_squared_error',
    random_state=42, n_jobs=-1
)
random_search.fit(X_train, y_train)
print("최적 하이퍼파라미터:", random_search.best_params_)

best_rf = random_search.best_estimator_
best_rmsle = round(rmsle(y_test, best_rf.predict(X_test)), 4)
print("튜닝 후 RandomForest RMSLE:", best_rmsle)

RandomForest RMSLE: 0.1871
LightGBM RMSLE: 0.1967
XGBoost RMSLE: 0.2068
최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': None}
튜닝 후 RandomForest RMSLE: 0.1859


### 정리

이 문제는 지금까지의 2유형과 달리 "제출 파일 생성"이 아니라 "여러 모델을 비교하고 최적화하는 과정 자체"가 목적이었습니다. RandomForest, LightGBM, XGBoost는 모두 트리 기반 앙상블 모델이라는 공통점이 있어 사용법(`.fit()`, `.predict()`)이 거의 동일하다는 것, RMSLE는 상대적 오차에 집중하는 지표라는 것, 그리고 RandomizedSearchCV로 하이퍼파라미터를 자동 탐색하면 사람이 값을 하나씩 바꿔가며 실험하지 않아도 더 나은 조합을 효율적으로 찾을 수 있다는 것을 확인했습니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

하이퍼파라미터 탐색 방법은 RandomizedSearchCV뿐 아니라 GridSearchCV로도 요구될 수 있습니다. 두 방법은 후보 조합을 무작위로 뽑는지 전부 시도하는지만 다를 뿐, 사용하는 코드의 구조는 거의 동일합니다.

**변형 예시: RandomizedSearchCV가 아니라 GridSearchCV로 탐색하라면**

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_small = {
    'n_estimators': [200, 300],
    'max_depth': [None, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_grid_small,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
print("GridSearchCV 최적 하이퍼파라미터:", grid_search.best_params_)

grid_pred = grid_search.best_estimator_.predict(X_test)
print("GridSearchCV 최적 모델 RMSLE:", round(rmsle(y_test, grid_pred), 4))

GridSearchCV 최적 하이퍼파라미터: {'max_depth': 10, 'n_estimators': 300}
GridSearchCV 최적 모델 RMSLE: 0.1874


`RandomizedSearchCV(param_distributions=..., n_iter=...)`가 `GridSearchCV(param_grid=...)`로 바뀌고, 무작위로 몇 개만 시도하는 `n_iter` 옵션이 사라진 것을 볼 수 있습니다. GridSearchCV는 후보 조합을 하나도 빠짐없이 전부 시도하므로, 후보 개수가 많으면(이번 예시처럼 후보를 줄이지 않으면 4×5×3×3=180가지) 탐색 시간이 크게 늘어날 수 있습니다. 그래서 여기서는 시연을 위해 후보 개수를 의도적으로 줄인 `param_grid_small`을 사용했습니다. 실전에서 GridSearchCV를 쓸 때는 후보값의 개수를 신중하게 제한해서, 제한 시간 안에 탐색이 끝날 수 있도록 하는 것이 중요합니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| RMSLE 계산 (사이킷런에 직접 없음) | `np.sqrt(mean_squared_log_error(y, pred))` | 로그 함수는 음수에서 정의되지 않으므로 예측값을 0 이상으로 클리핑 필요 |
| LightGBM 회귀모델 | `lgb.LGBMRegressor()` | RandomForest와 동일한 `.fit()`/`.predict()` 인터페이스 |
| XGBoost 회귀모델 | `xgb.XGBRegressor()` | 마찬가지로 동일한 인터페이스, verbosity로 로그 제어 |
| 여러 모델의 성능을 표로 비교 | `pd.DataFrame({...}).sort_values()` | 오차 지표는 오름차순 정렬 시 맨 위가 최적 모델 |
| 하이퍼파라미터를 무작위로 탐색 | `RandomizedSearchCV(param_distributions=, n_iter=)` | 후보가 많을 때 GridSearchCV보다 빠르게 좋은 조합을 탐색 |
| 하이퍼파라미터를 전수 탐색 | `GridSearchCV(param_grid=)` | 모든 조합을 시도하므로 후보 개수 관리가 중요 |
| 탐색 후 가장 좋은 모델 바로 사용 | `search.best_estimator_` | 이미 최적 파라미터로 학습까지 완료된 모델 객체 |